In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.stats import rankdata

In [ ]:
DATA_FILES = [
    (1,  'ha-1sec-full-rl-v4.pqt'),
    (2,  'ha-2sec-full-rl-v4.pqt'),
    (3,  'ha-3sec-full-rl-v4.pqt'),
    (4,  'ha-4sec-full-rl-v4.pqt'),
    (8,  'ha-8sec-full-rl-v4.pqt'),
    (16, 'ha-16sec-full-rl-v4.pqt'),
]
OUT_DIR = 'e2_out'
L = 32
MIN_LEN = 4
MIN_RANGE_PTS = 0.5
TICK = 0.0625
POOL_CAP = 300_000
PAIR_CAP = 150_000
TEST_FRAC = 0.2
NUM_ROUNDS = 150
PARAMS = {
    'objective': 'binary',
    'learning_rate': 0.1,
    'num_leaves': 63,
    'min_data_in_leaf': 100,
    'verbose': -1,
    'seed': 7,
}

os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(7)

In [ ]:
def load_shapes(path):
    df = pd.read_parquet('../'+path, columns=['date', 'Close', 'jmaD1'])
    date = df['date'].to_numpy()
    close = df['Close'].to_numpy(np.float64)
    s = pd.Series(np.sign(df['jmaD1'].to_numpy())).replace(0.0, np.nan)
    s = s.groupby(date).ffill().groupby(date).bfill().to_numpy()
    m = len(df)
    new_day = np.empty(m, dtype=bool); new_day[0] = True
    new_day[1:] = date[1:] != date[:-1]
    flip = np.empty(m, dtype=bool); flip[0] = True
    flip[1:] = s[1:] != s[:-1]
    starts = np.flatnonzero(new_day | flip)
    ends = np.r_[starts[1:], m]
    lens = ends - starts
    keep = np.flatnonzero(lens >= MIN_LEN)
    if len(keep) > POOL_CAP:
        keep = rng.choice(keep, POOL_CAP, replace=False)
    xs = np.linspace(0.0, 1.0, L)
    shapes = np.empty((len(keep), L), np.float32)
    qs = np.empty(len(keep), np.float32)
    days = date[starts[keep]]
    ok = np.ones(len(keep), dtype=bool)
    for i, k in enumerate(keep):
        seg = close[starts[k]:ends[k]]
        lo = seg.min()
        r = seg.max() - lo
        if r < MIN_RANGE_PTS:
            ok[i] = False
            continue
        shapes[i] = np.interp(xs, np.linspace(0.0, 1.0, lens[k]), seg - lo) / r
        qs[i] = TICK / r
    return shapes[ok], qs[ok], days[ok], lens[keep][ok]


def auc(y, p):
    r = rankdata(p)
    n1 = int(y.sum())
    n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)

In [ ]:
frames = {}
for n, path in DATA_FILES:
    frames[n] = load_shapes(path)
    print(f"n={n:>2}s segments kept: {len(frames[n][0]):,}", flush=True)

In [ ]:
ns = sorted(frames)
K = len(ns)
A_mat = np.full((K, K), np.nan)
rows = []
for i in range(K):
    for j in range(i + 1, K):
        sa, qa, da, la = frames[ns[i]]
        sb, qb, db, lb = frames[ns[j]]
        ba = np.minimum(la, 48) * 1000 + np.floor(np.log2(1.0 / qa) * 4.0).astype(np.int64)
        bb = np.minimum(lb, 48) * 1000 + np.floor(np.log2(1.0 / qb) * 4.0).astype(np.int64)
        ia_l, ib_l = [], []
        for b in np.intersect1d(np.unique(ba), np.unique(bb)):
            ca = np.flatnonzero(ba == b)
            cb = np.flatnonzero(bb == b)
            k = min(len(ca), len(cb))
            if k:
                ia_l.append(rng.choice(ca, k, replace=False))
                ib_l.append(rng.choice(cb, k, replace=False))
        ia = np.concatenate(ia_l)
        ib = np.concatenate(ib_l)
        if len(ia) > PAIR_CAP:
            sel = rng.choice(len(ia), PAIR_CAP, replace=False)
            ia, ib = ia[sel], ib[sel]
        take = len(ia)
        X = np.vstack([sa[ia], sb[ib]])
        y = np.r_[np.zeros(take, np.int8), np.ones(take, np.int8)]
        day = np.concatenate([da[ia], db[ib]])
        ud = np.unique(day)
        te_days = rng.choice(ud, int(len(ud) * TEST_FRAC), replace=False)
        te = np.isin(day, te_days)
        tr = ~te
        mdl = lgb.train(PARAMS, lgb.Dataset(X[tr], label=y[tr]), num_boost_round=NUM_ROUNDS)
        a_m = auc(y[te], mdl.predict(X[te]))
        A_mat[i, j] = A_mat[j, i] = a_m
        rows.append((ns[i], ns[j], take, a_m))
        print(f"{ns[i]:>2}s vs {ns[j]:>2}s  joint-matched n/class={take:,}  AUC={a_m:.4f}", flush=True)

res = pd.DataFrame(rows, columns=['n_a', 'n_b', 'per_class', 'auc_joint'])
res.to_csv(os.path.join(OUT_DIR, 'e2_pairs_joint.csv'), index=False)
np.fill_diagonal(A_mat, 0.5)

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(A_mat, vmin=0.5, vmax=max(0.55, float(np.nanmax(A_mat))), cmap='viridis')
ax.set_xticks(range(K), [f"{n}s" for n in ns])
ax.set_yticks(range(K), [f"{n}s" for n in ns])
ax.set_title('AUC L+range matched')
for a in range(K):
    for b in range(K):
        ax.text(b, a, f"{A_mat[a, b]:.3f}", ha='center', va='center', color='w', fontsize=8)
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'auc_matrix_joint.png'), dpi=130)
plt.show()